# Final Non-Demographic Visualizations

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib.colors as mcolors # Import mcolors for LinearSegmentedColormap
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.font_manager as fm
import os


In [ ]:
df1 = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/final_data/21_23_full_final.csv')
df2 = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/final_data/23_25_full_final.csv')
merged = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/merged/merged_full.csv')

In [ ]:
df1_demo = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/final_data/21_23_demo_final.csv')
df2_demo = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/final_data/23_25_demo_final.csv')
merged_demo = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/merged/merged_demo.csv')

In [ ]:

# Before your groupby step:
# df1['is_non-white_household'] = df1['race_simplified'].map(non_white_mapping) # This line should be already there

# Fill NaN values in 'age_restricted' with False
df1['age_restricted'] = df1['age_restricted'].fillna('No')
df2['age_restricted'] = df2['age_restricted'].fillna('No')
merged['age_restricted'] = merged['age_restricted'].fillna('No')

In [ ]:
# Subset for age-restricted properties
age_restricted_df1 = df1[df1['age_restricted'].isin(['55', '55+', '62'])].copy()
age_restricted_df2 = df2[df2['age_restricted'].isin(['55', '55+', '62'])].copy()
age_restricted_merged = merged[merged['age_restricted'].isin(['55', '55+', '62'])].copy()


# Subset for non-age-restricted properties
non_age_restricted_df1 = df1[df1['age_restricted'] == 'No'].copy()
non_age_restricted_df2 = df2[df2['age_restricted'] == 'No'].copy()
non_age_restricted_merged = merged[merged['age_restricted'] == 'No'].copy()

# formatting

## save plot

In [ ]:

def save_plot(filename, subfolder="BQ1", dpi=300):
    """
    Save the current matplotlib figure to a specific BQ subfolder inside the PITNE Visualizations directory.

    Parameters:
        filename (str): Name of the file (e.g., "plot1.png").
        subfolder (str): One of "BQ1" through "BQ5" specifying the subfolder (default: "BQ1").
        dpi (int): DPI for the saved figure (default: 300).
    """
    base_folder = "/Users/kaylamullen/Desktop/PITNE/Visualizations"
    
    # Validate subfolder
    valid_subfolders = {"BQ1", "BQ2", "BQ3", "BQ4", "BQ5"}
    if subfolder not in valid_subfolders:
        raise ValueError(f"Invalid subfolder '{subfolder}'. Choose one of: {', '.join(valid_subfolders)}")
    
    # Create full folder path
    folder_path = os.path.join(base_folder, subfolder)
    os.makedirs(folder_path, exist_ok=True)
    
    # Build full file path
    filepath = os.path.join(folder_path, filename)
    
    # Save the figure
    plt.savefig(filepath, dpi=dpi, bbox_inches='tight')
    print(f"Plot saved to: {filepath}")


## DPI

In [ ]:
# Set default DPI for all figures
plt.rcParams['figure.dpi'] = 250

## fonts

In [ ]:
# Step 1: Path to the .ttf file
font_path = 'Lexend-VariableFont_wght.ttf'

# Step 2: Add font to matplotlib's font manager
fm.fontManager.addfont(font_path)

# Step 3: Get the font name from the file
lexend_prop = fm.FontProperties(fname=font_path)
lexend_name = lexend_prop.get_name()
print("Lexend font name:", lexend_name)

# Step 4: Set globally for current session
from matplotlib import rcParams

# Force all text elements to use the actual FontProperties, not just font name
rcParams['font.family'] = lexend_prop.get_name()
rcParams['font.sans-serif'] = [lexend_prop.get_name()]
rcParams['text.usetex'] = False  # Make sure TeX isn't interfering


sns.set_theme(style='whitegrid')


In [ ]:
# Helper to apply Lexend font across all elements
def use_lexend_globally():

    font_path = 'Lexend-VariableFont_wght.ttf'
    lexend_prop = fm.FontProperties(fname=font_path)
    lexend_name = lexend_prop.get_name()

    # Set global font properties
    rcParams['font.family'] = lexend_name
    rcParams['font.sans-serif'] = [lexend_name]
    rcParams['axes.titlesize'] = 14
    rcParams['axes.labelsize'] = 12
    rcParams['xtick.labelsize'] = 10
    rcParams['ytick.labelsize'] = 10
    rcParams['legend.fontsize'] = 10

    print(f"Using font: {lexend_name}")

use_lexend_globally()


In [ ]:
df1.columns

## Colors!

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define your custom hex palette
custom_palette = [
    "#1f1e40",  # Space Cadet
    "#005757",  # Dark Slate Gray
    "#6da59b",  # Cambridge Blue
    "#daf3df",  # Honeydew
    "#ffd9bf",  # Apricot
    "#ff8f80",  # Coral Pink
    "#f25757",  # Bittersweet
    "#721817",  # Falu Red
    "#dc7500"   # Ochre
]

# Set as global Seaborn palette
sns.set_palette(custom_palette)

# Optional: Also set Matplotlib default color cycle
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=custom_palette)


In [ ]:
chapa_colors = ["#1F1E40", "#005757", "#DAF3DF", "#FF8F80"]

In [ ]:
race_color_map = {
    'White': '#005757',
    'Black or African American': '#DC7500',
    'Hispanic or Latino (of any race)': '#1F1E40',
    'Middle Eastern or North African': '#6DA59B',
    'Asian': '#FF8F80',
    'Native American or Alaskan Native':'#DAF3DF',
    'Multiple Races/Ethnicities': '#721817',
    'Choose not to answer/Missing/Other': '#F25757'
}


In [ ]:

def race_countplot(data, x='race_simplified', **kwargs):
    """
    Wrapper around sns.countplot that auto-applies the race color palette.
    
    Parameters:
        data (pd.DataFrame): The dataframe to plot.
        x (str): Column to plot (defaults to 'race_simplified').
        kwargs: Additional keyword arguments passed to sns.countplot.
    """
    if x == 'race_simplified':
        palette = race_color_map
        order = list(race_color_map.keys())
    else:
        palette = kwargs.pop('palette', None)
        order = kwargs.pop('order', None)
    
    ax = sns.countplot(data=data, x=x, palette=palette, order=order, **kwargs)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    return ax


# BQ1

## HH of Color vs White HH Town Distribution

### mapping

In [ ]:
non_white_mapping = {
    'Hispanic or Latino (of all races)' : 1, 
    'Asian' : 1, 
    'Black or African American' : 1, 
    'Middle Eastern or North African' : 1,
    'Native American or Alaskan Native' : 1,
    'Choose not to answer/Missing/Other' : -1,
    'White' : 0,
    'Multiple Races/Ethnicities' : 1,
}

df1['is_non-white_household'] = df1['race_simplified'].map(non_white_mapping)
df2['is_non-white_household'] = df2['race_simplified'].map(non_white_mapping)
merged['is_non-white_household'] = merged ['race_simplified'].map(non_white_mapping)

In [ ]:
by_town_non_white1 = df1.groupby(['is_non-white_household', 'property_town']).size().reset_index(name='num_applications')
by_town_non_white2 = df2.groupby(['is_non-white_household', 'property_town']).size().reset_index(name='num_applications')
by_town_non_white_merged = merged.groupby(['is_non-white_household', 'property_town']).size().reset_index(name='num_applications')

int_to_label = {
    -1: 'Unknown',
    0: 'White Household (non-hispanic)',
    1: 'Household of Color'
}

by_town_non_white1['non-white_label'] = by_town_non_white1['is_non-white_household'].map(int_to_label)
by_town_non_white2['non-white_label'] = by_town_non_white2['is_non-white_household'].map(int_to_label)
by_town_non_white_merged['non-white_label'] = by_town_non_white_merged['is_non-white_household'].map(int_to_label)

In [ ]:
custom_palette = {
    0:  "#005757",  # White Household
    1: "#FF8F80"   # Household of Color
}

### by number of applications

In [ ]:
# Step 1: Filter data
plot_data_merged = merged[merged['is_non-white_household'] >= 0]

# Step 2: Sort towns alphabetically
town_order_merged = sorted(merged['property_town'].unique())

# Step 3: Create the plot
plt.figure(figsize=(10, 12))
ax = sns.countplot(
    data=plot_data_merged,
    y='property_town',
    hue='is_non-white_household',
    order=town_order_merged,
    palette=custom_palette
)

plt.title("Applications per Town by Household of Color (2021-2025)")
plt.ylabel("Town")  # 👈 X-axis label
plt.xlabel("Number of Applications")  # 👈 Y-axis label

# Step 4: Set legend labels manually
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles=handles, labels=['White Household', 'Household of Color'], title='Household Type')

plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Filter data
plot_data1 = df1[df1['is_non-white_household'] >= 0]

# Step 2: Sort towns alphabetically
town_order1 = sorted(df1['property_town'].unique())

# Step 3: Create the plot
plt.figure(figsize=(10, 12))
ax = sns.countplot(
    data=plot_data1,
    y='property_town',
    hue='is_non-white_household',
    order=town_order1,
    palette=custom_palette
)

plt.title("Applications per Town by Household of Color")
plt.ylabel("Town")  # 👈 X-axis label
plt.xlabel("Number of Applications")  # 👈 Y-axis label

# Step 4: Set legend labels manually
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles=handles, labels=['White Household', 'Household of Color'], title='Household Type')

plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Filter data
plot_data2 = df2[df2['is_non-white_household'] >= 0]

# Step 2: Sort towns alphabetically
town_order2 = sorted(df2['property_town'].unique())

# Step 3: Create the plot
plt.figure(figsize=(10, 12))
ax = sns.countplot(
    data=plot_data2,
    y='property_town',
    hue='is_non-white_household',
    order=town_order2,
    palette=custom_palette
)

plt.title("Applications per Town by Household of Color")
plt.ylabel("Town")  # 👈 X-axis label
plt.xlabel("Number of Applications")  # 👈 Y-axis label

# Step 4: Set legend labels manually
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles=handles, labels=['White Household', 'Household of Color'], title='Household Type')

plt.tight_layout()
plt.show()

### Towns by households of color (percentage %)

In [ ]:
prop_table1 = by_town_non_white1.copy()
race_group_totals1 = prop_table1.groupby('is_non-white_household')['num_applications'].transform('sum')
prop_table1['percent_of_group'] = (prop_table1['num_applications'] / race_group_totals1 * 100).round(1)

prop_table2 = by_town_non_white2.copy()
race_group_totals2 = prop_table2.groupby('is_non-white_household')['num_applications'].transform('sum')
prop_table2['percent_of_group'] = (prop_table2['num_applications'] / race_group_totals2 * 100).round(1)

In [ ]:
# Filter out unknowns (-1)
plot_data1 = prop_table1[prop_table1['is_non-white_household'] >= 0].copy()
total_applications = plot_data1['num_applications'].sum()
print(f"Total number of applications: {total_applications}")

In [ ]:


plt.figure(figsize=(10, 12))
sns.barplot(
    data=plot_data1,
    y='property_town',
    x='percent_of_group',
    hue='is_non-white_household',
    order=town_order1,
    palette=custom_palette
)

plt.title('2021-2023 Percent (%) of Applications to Each Town by Households of Color')
plt.ylabel('Property Town')
plt.xlabel('Percent of Applications from White Households vs Households of Color')
plt.legend(handles=handles, labels=['White Household', 'Household of Color'], title='Household Type')
plt.tight_layout()
plt.show()


In [ ]:
# Filter out unknowns (-1)
plot_data2 = prop_table2[prop_table2['is_non-white_household'] >= 0].copy()


plt.figure(figsize=(10, 12))
sns.barplot(
    data=plot_data2,
    y='property_town',
    x='percent_of_group',
    hue='is_non-white_household',
    order=town_order2,
    palette=custom_palette
)

plt.title('2023-2025 Percent (%) of Applications to Each Town by Households of Color')
plt.ylabel('Property Town')
plt.xlabel('Percent of Applications from White Households vs Households of Color')
plt.legend(handles=handles, labels=['White Household', 'Household of Color'], title='Household Type')
plt.tight_layout()
plt.show()


### Pie Chart

In [ ]:
def top_n_plus_other(df, n=8):
    df = df.sort_values('percent_of_group', ascending=False).reset_index(drop=True)
    if len(df) <= n:
        return df[['property_town', 'percent_of_group']]
    
    top = df.iloc[:n]
    other = df.iloc[n:]
    other_sum = other['percent_of_group'].sum()
    
    other_row = pd.DataFrame({'property_town': ['Other'], 'percent_of_group': [other_sum]})
    return pd.concat([top[['property_town', 'percent_of_group']], other_row], ignore_index=True)


In [ ]:
for group_val, group_name in zip([0, 1], ['White Households', 'Households of Color']):
    group_data = plot_data1[plot_data1['is_non-white_household'] == group_val]
    pie_data = top_n_plus_other(group_data, n=10)

    plt.figure(figsize=(8, 8))
    plt.pie(
        pie_data['percent_of_group'],
        labels=pie_data['property_town'],
        autopct='%1.1f%%',
        startangle=140
    )
    plt.title(f'{group_name} - Top Towns by Application Share (prop_table1)')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [ ]:
for group_val, group_name in zip([0, 1], ['White Households', 'Households of Color']):
    group_data = plot_data2[plot_data2['is_non-white_household'] == group_val]
    pie_data = top_n_plus_other(group_data, n=15)

    plt.figure(figsize=(8, 8))
    plt.pie(
        pie_data['percent_of_group'],
        labels=pie_data['property_town'],
        autopct='%1.1f%%',
        startangle=140
    )
    plt.title(f'{group_name} - Top 10 Towns by Application Share (2023-2025)')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


## Property town by Income

### All Applications

In [ ]:
# Step 1: Identify the top 10 towns by number of applications
top_towns1 = df1['property_town'].value_counts().head(10).index
top_towns2 = df2['property_town'].value_counts().head(10).index
top_towns_merged = merged['property_town'].value_counts().head(10).index


# Step 2: Filter data to only include those towns
top_towns_filtered1 = df1[df1['property_town'].isin(top_towns1)]
top_towns_filtered2 = df2[df2['property_town'].isin(top_towns2)]
top_towns_filtered_merged = merged[merged['property_town'].isin(top_towns_merged)]

In [ ]:
income_median_order_merged = (
    top_towns_filtered_merged.groupby('property_town')['hh_income']
    .median()
    .sort_values(ascending=False)
    .index
)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered_merged,
    x='property_town',
    y='hh_income',
    order=income_median_order_merged,
    color="#FF8F80"
    
)

plt.title('2021-2025 Distribution of Household Income in Top 10 Applied to Towns')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Income_top10_21_25", subfolder="BQ1")
plt.show()



In [ ]:
# zoom in on merged (exclude outliers)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered_merged,
    x='property_town',
    y='hh_income',
    order=income_median_order_merged,
    color="#FF8F80"
    
)
plt.ylim(0, 200000)
plt.title('2021-2025 Distribution of Household Income in Top 10 Applied to Towns')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Income_top10_21_25_zoom")
plt.show()

In [ ]:
income_median_order1 = (
    top_towns_filtered1.groupby('property_town')['hh_income']
    .median()
    .sort_values(ascending=False)
    .index
)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered1,
    x='property_town',
    y='hh_income',
    order=income_median_order1,
    color="#FF8F80"
    
)

plt.title('2021-2023 Distribution of Household Income in Top 10 Applied to Towns')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
income_median_order2 = (
    top_towns_filtered2.groupby('property_town')['hh_income']
    .median()
    .sort_values(ascending=False)
    .index
)

In [ ]:

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered2,
    x='property_town',
    y='hh_income',
    order=income_median_order2,
    color="#FF8F80"
)

plt.ylim(0, 200000)
plt.title('2023-2025 Distribution of Household Income in Top 10 Towns')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Age restricted only

In [ ]:
# Drop rows with missing income or town
plot_data = age_restricted_merged.dropna(subset=['hh_income', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)

plt.title('2021-2025 Income Distribution by Town (Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Income_AR_21_25")
plt.show()

In [ ]:
# Drop rows with missing income or town
plot_data = age_restricted_df1.dropna(subset=['hh_income', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)

plt.title('2021-2023 Income Distribution by Town (Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Drop rows with missing income or town
plot_data = age_restricted_df2.dropna(subset=['hh_income', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)

plt.title('2023-2025 Income Distribution by Town (Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
age_restricted_df1.shape

### Non-Age Restricted

In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_merged.dropna(subset=['hh_income', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)

plt.title('2021-2025 Income Distribution by Town (Non Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_df1.dropna(subset=['hh_income', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)

plt.title('2021-2023 Income Distribution by Town (Non Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_merged.dropna(subset=['hh_income', 'property_town'])

# ✅ Get top 10 towns by number of applications
top_towns = plot_data['property_town'].value_counts().nlargest(10).index

# ✅ Filter to include only those towns
plot_data = plot_data[plot_data['property_town'].isin(top_towns)]

# ✅ Sort these towns by median income for cleaner boxplot order
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)
plt.ylim(0, 200000)
plt.title('2021–2025 Income Distribution by Town (Top 10 Non-Age-Restricted Properties)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Income_NAR_top10_21_25")
plt.show()


In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_df1.dropna(subset=['hh_income', 'property_town'])

# ✅ Get top 10 towns by number of applications
top_towns = plot_data['property_town'].value_counts().nlargest(10).index

# ✅ Filter to include only those towns
plot_data = plot_data[plot_data['property_town'].isin(top_towns)]

# ✅ Sort these towns by median income for cleaner boxplot order
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)
plt.ylim(0, 200000)
plt.title('2021–2023 Income Distribution by Town (Top 10 Non-Age-Restricted Properties)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_df2.dropna(subset=['hh_income', 'property_town'])

# ✅ Get top 10 towns by number of applications
top_towns = plot_data['property_town'].value_counts().nlargest(10).index

# ✅ Filter to include only those towns
plot_data = plot_data[plot_data['property_town'].isin(top_towns)]

# ✅ Sort these towns by median income for cleaner boxplot order
median_order = plot_data.groupby('property_town')['hh_income'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_income',
    order=median_order,
    color="#FF8F80"
)
# plt.ylim(0, 200000)
plt.title('2023–2025 Income Distribution by Town (Top 10 Non-Age-Restricted Properties)')
plt.xlabel('Town')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Assets by Town

### All Applications

In [ ]:
median_order = (
    top_towns_filtered_merged.groupby('property_town')['hh_assets']
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered_merged,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
    
)

plt.title('2021-2023 Distribution of Household Assets in Top 10 Towns')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
median_order = (
    top_towns_filtered1.groupby('property_town')['hh_assets']
    .median()
    .sort_values(ascending=False)
    .index
)
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered1,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
    
)

plt.title('2021-2025 Distribution of Household Assets in Top 10 Towns')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Assets_top10_21_25")
plt.show()

In [ ]:
median_order = (
    top_towns_filtered2.groupby('property_town')['hh_assets']
    .median()
    .sort_values(ascending=False)
    .index
)
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=top_towns_filtered2,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
    
)

plt.title('2023-2025 Distribution of Household Assets in Top 10 Towns')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### age restricted

In [ ]:
# Drop rows with missing income or town
plot_data = age_restricted_merged.dropna(subset=['hh_assets', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_assets'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
)

plt.ylim(0, 500000)
plt.title('2021-2025 Assets Distribution by Town (Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Assets_AR_21_25")
plt.show()

In [ ]:
# Drop rows with missing income or town
plot_data = age_restricted_df1.dropna(subset=['hh_assets', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_assets'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
)

plt.ylim(0, 500000)
plt.title('2021-2023 Assets Distribution by Town (Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Drop rows with missing income or town
plot_data = age_restricted_df2.dropna(subset=['hh_assets', 'property_town'])

# Optional: sort towns by median income for better readability
median_order = plot_data.groupby('property_town')['hh_assets'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
)
plt.ylim(0, 500000)
plt.title('2023-2025 Assets Distribution by Town (Age-Restricted Properties Only)')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### non age restricted

In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_merged.dropna(subset=['hh_assets', 'property_town'])

# ✅ Get top 10 towns by number of applications
top_towns = plot_data['property_town'].value_counts().nlargest(10).index

# ✅ Filter to include only those towns
plot_data = plot_data[plot_data['property_town'].isin(top_towns)]

# ✅ Sort these towns by median income for cleaner boxplot order
median_order = plot_data.groupby('property_town')['hh_assets'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
)
plt.ylim(0, 155000)
plt.title('2021–2025 Asset Distribution by Town (Top 10 Non-Age-Restricted Properties)')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("HH_Assets_NAR_top10_21_25")
plt.show()


In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_df1.dropna(subset=['hh_assets', 'property_town'])

# ✅ Get top 10 towns by number of applications
top_towns = plot_data['property_town'].value_counts().nlargest(10).index

# ✅ Filter to include only those towns
plot_data = plot_data[plot_data['property_town'].isin(top_towns)]

# ✅ Sort these towns by median income for cleaner boxplot order
median_order = plot_data.groupby('property_town')['hh_assets'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
)
plt.ylim(0, 155000)
plt.title('2021–2023 Asset Distribution by Town (Top 10 Non-Age-Restricted Properties)')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Drop rows with missing income or town
plot_data = non_age_restricted_df2.dropna(subset=['hh_assets', 'property_town'])

# ✅ Get top 10 towns by number of applications
top_towns = plot_data['property_town'].value_counts().nlargest(10).index

# ✅ Filter to include only those towns
plot_data = plot_data[plot_data['property_town'].isin(top_towns)]

# ✅ Sort these towns by median income for cleaner boxplot order
median_order = plot_data.groupby('property_town')['hh_assets'].median().sort_values(ascending=False).index

# Create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x='property_town',
    y='hh_assets',
    order=median_order,
    color="#FF8F80"
)
plt.ylim(0, 155000)
plt.title('2023–2025 Asset Distribution by Town (Top 10 Non-Age-Restricted Properties)')
plt.xlabel('Town')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Towns by Age

### All applications

In [ ]:
# Drop rows with missing age or town
plot_data1 = merged.dropna(subset=['age', 'property_town'])

# Optional: limit to top 10 towns by number of applications
top_towns1 = plot_data1['property_town'].value_counts().nlargest(10).index
plot_data1 = plot_data1[plot_data1['property_town'].isin(top_towns1)]

# Sort towns by median age
median_order1 = plot_data1.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data1,
    x='property_town',
    y='age',
    order=median_order1,
    color="#FF8F80"
)
plt.title('Age Distribution Across Top 10 Towns (2021-2025)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("Age_top10_21_25")
plt.show()

In [ ]:
# Drop rows with missing age or town
plot_data1 = df1.dropna(subset=['age', 'property_town'])

# Optional: limit to top 10 towns by number of applications
top_towns1 = plot_data1['property_town'].value_counts().nlargest(10).index
plot_data1 = plot_data1[plot_data1['property_town'].isin(top_towns1)]

# Sort towns by median age
median_order1 = plot_data1.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data1,
    x='property_town',
    y='age',
    order=median_order1,
    color="#005757"
)
plt.title('Age Distribution by Town (2021-2023)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Ensure age is numeric
df2['age'] = pd.to_numeric(df2['age'], errors='coerce')

# Drop rows with missing age or town
plot_data2 = df2.dropna(subset=['age', 'property_town'])

# Optional: limit to top 10 towns
top_towns2 = plot_data2['property_town'].value_counts().nlargest(10).index
plot_data2 = plot_data2[plot_data2['property_town'].isin(top_towns2)]

# Sort towns by median age
median_order2 = plot_data2.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data2,
    x='property_town',
    y='age',
    order=median_order2,
    color="#005757"
)
plt.title('Age Distribution by Town (2023-2025)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### non-age restricted

In [ ]:
# Ensure age is numeric
non_age_restricted_merged['age'] = pd.to_numeric(non_age_restricted_merged['age'], errors='coerce')

# Drop rows with missing age or town
plot_data1 = non_age_restricted_merged.dropna(subset=['age', 'property_town'])

# Limit to top 10 towns by number of applications
top_towns1 = plot_data1['property_town'].value_counts().nlargest(10).index
plot_data1 = plot_data1[plot_data1['property_town'].isin(top_towns1)]

# Sort by median age
median_order1 = plot_data1.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data1,
    x='property_town',
    y='age',
    order=median_order1,
    color="#FF8F80"
)
plt.title('2021-2025 Age Distribution Across Top 10 Towns (Non-Age-Restricted)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("Age_NAR_top10_21_25")
plt.show()


In [ ]:
# Ensure age is numeric
non_age_restricted_df1['age'] = pd.to_numeric(non_age_restricted_df1['age'], errors='coerce')

# Drop rows with missing age or town
plot_data1 = non_age_restricted_df1.dropna(subset=['age', 'property_town'])

# Limit to top 10 towns by number of applications
top_towns1 = plot_data1['property_town'].value_counts().nlargest(10).index
plot_data1 = plot_data1[plot_data1['property_town'].isin(top_towns1)]

# Sort by median age
median_order1 = plot_data1.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data1,
    x='property_town',
    y='age',
    order=median_order1,
    color="#005757"
)
plt.title('2021-2023 Age Distribution by Town (Non-Age-Restricted)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Ensure age is numeric
non_age_restricted_df2['age'] = pd.to_numeric(non_age_restricted_df2['age'], errors='coerce')

# Drop rows with missing age or town
plot_data2 = non_age_restricted_df2.dropna(subset=['age', 'property_town'])

# Limit to top 10 towns by number of applications
top_towns2 = plot_data2['property_town'].value_counts().nlargest(10).index
plot_data2 = plot_data2[plot_data2['property_town'].isin(top_towns2)]

# Sort by median age
median_order2 = plot_data2.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data2,
    x='property_town',
    y='age',
    order=median_order2,
    color="#005757"
)
plt.title('2023-2025 Age Distribution by Town (Non-Age-Restricted)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### age restricted

In [ ]:
# Ensure 'age' is numeric
age_restricted_merged['age'] = pd.to_numeric(age_restricted_merged['age'], errors='coerce')

# Drop rows with missing age or town
plot_data1 = age_restricted_merged.dropna(subset=['age', 'property_town'])

# Top 10 towns by number of applications
top_towns1 = plot_data1['property_town'].value_counts().nlargest(10).index
plot_data1 = plot_data1[plot_data1['property_town'].isin(top_towns1)]

# Sort towns by median age
median_order1 = plot_data1.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data1,
    x='property_town',
    y='age',
    order=median_order1,
    color="#FF8F80"  # Dark, high-contrast color
)
plt.title('2021-2025 Age Distribution by Town (Age-Restricted)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot("Age_AR_top10_21_25")
plt.show()


In [ ]:
# Ensure 'age' is numeric
age_restricted_df1['age'] = pd.to_numeric(age_restricted_df1['age'], errors='coerce')

# Drop rows with missing age or town
plot_data1 = age_restricted_df1.dropna(subset=['age', 'property_town'])

# Top 10 towns by number of applications
top_towns1 = plot_data1['property_town'].value_counts().nlargest(10).index
plot_data1 = plot_data1[plot_data1['property_town'].isin(top_towns1)]

# Sort towns by median age
median_order1 = plot_data1.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data1,
    x='property_town',
    y='age',
    order=median_order1,
    color="#005757"  # Dark, high-contrast color
)
plt.title('2021-2023 Age Distribution by Town (Age-Restricted)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Ensure 'age' is numeric
age_restricted_df2['age'] = pd.to_numeric(age_restricted_df2['age'], errors='coerce')

# Drop rows with missing age or town
plot_data2 = age_restricted_df2.dropna(subset=['age', 'property_town'])

# Top 10 towns by number of applications
top_towns2 = plot_data2['property_town'].value_counts().nlargest(10).index
plot_data2 = plot_data2[plot_data2['property_town'].isin(top_towns2)]

# Sort towns by median age
median_order2 = plot_data2.groupby('property_town')['age'].median().sort_values(ascending=False).index

# Plot
plt.figure(figsize=(15, 6))
sns.boxplot(
    data=plot_data2,
    x='property_town',
    y='age',
    order=median_order2,
    color="#005757"  # Calm, clear contrast
)
plt.title('2023-2025 Age Distribution by Town (Age-Restricted)')
plt.xlabel('Town')
plt.ylabel('Applicant Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# BQ2

## Income by Race/Ethnicity

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo,
    x='race_simplified',
    y='hh_income',
    palette=race_color_map,
    order=list(race_color_map.keys())
)

plt.title('Household Income Distribution by Race/Ethnicity (2021-2025)')
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Income_Race_21_25', "BQ2")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo,
    x='race_simplified',
    y='hh_income',
    palette=race_color_map,
    order=list(race_color_map.keys())
)
plt.title('Household Income Distribution by Race/Ethnicity Zoomed In (2021-2025)')
plt.ylim(0, 200000)
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Income_Race_21_25_Zoom', "BQ2")
plt.show()

In [ ]:
# Step 1: Label the datasets
df1_demo['dataset_period'] = '2021–2023'
df2_demo['dataset_period'] = '2023–2025'

# Step 2: Merge
merged_demo1 = pd.concat([df1_demo, df2_demo], ignore_index=True)

# Step 3: Define custom color map for the periods
period_color_map = {
    '2021–2023': '#F25757',
    '2023–2025': '#daf3df'
}

In [ ]:
# Step 4: Plot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo1,
    x='race_simplified',
    y='hh_income',
    hue='dataset_period',
    palette=period_color_map,
    order=list(race_color_map.keys())  # keeps race order consistent
)
plt.title('Household Income Distribution by Race/Ethnicity (LITE vs FULL)')
plt.ylim(0, 200000)
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Income_Race_Period_Comparison', "BQ2")
plt.show()

## Assets by Race/Ethnicity

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo,
    x='race_simplified',
    y='hh_assets',
    palette=race_color_map,
    order=list(race_color_map.keys())
)
plt.title('Household Asset Distribution by Race/Ethnicity (2021-2025)')
# plt.ylim(0, 200000)
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Asset_Race_21_25', "BQ2")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo,
    x='race_simplified',
    y='hh_assets',
    palette=race_color_map,
    order=list(race_color_map.keys())
)
plt.title('Household Asset Distribution by Race/Ethnicity Zoomed In (2021-2025)')
plt.ylim(0, 300000)
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Income ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Asset_Race_21_25_Zoom', "BQ2")
plt.show()

In [ ]:
# Step 4: Plot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo1,
    x='race_simplified',
    y='hh_assets',
    hue='dataset_period',
    palette=period_color_map,
    order=list(race_color_map.keys())  # keeps race order consistent
)
plt.title('Household Asset Distribution by Race/Ethnicity (LITE vs FULL)')
# plt.ylim(0, 200000)
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Assets_Race_Period_Comparison', "BQ2")
plt.show()

In [ ]:
# Step 4: Plot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=merged_demo1,
    x='race_simplified',
    y='hh_assets',
    hue='dataset_period',
    palette=period_color_map,
    order=list(race_color_map.keys())  # keeps race order consistent
)
plt.title('Household Asset Distribution by Race/Ethnicity (LITE vs FULL)')
plt.ylim(0, 300000)
plt.xlabel('Race/Ethnicity')
plt.ylabel('Household Assets ($)')
plt.xticks(rotation=45)
plt.tight_layout()
save_plot('HH_Assets_Race_Period_Comparison_Zoom', "BQ2")
plt.show()

In [ ]:
median_incomes = df1_demo.groupby('race_simplified')['hh_income'].median().sort_values(ascending=False)
print(median_incomes)


In [ ]:
median_incomes = df2_demo.groupby('race_simplified')['hh_income'].median().sort_values(ascending=False)
print(median_incomes)


In [ ]:
median_incomes = merged_demo.groupby('race_simplified')['hh_income'].median().sort_values(ascending=False)
print(median_incomes)


In [ ]:
median_incomes = df1_demo['hh_income'].median()
print(median_incomes)

In [ ]:
median_incomes = df2_demo['hh_income'].median()
print(median_incomes)

In [ ]:
median_incomes = age_restricted_df1['hh_income'].median()
print(median_incomes)

In [ ]:
median_incomes = non_age_restricted_df1['hh_income'].median()
print(median_incomes)

In [ ]:
median_incomes = age_restricted_df2['hh_income'].median()
print(median_incomes)

In [ ]:
median_incomes = non_age_restricted_df2['hh_income'].median()
print(median_incomes)

In [ ]:
median_incomes = df1_demo['hh_assets'].median()
print(median_incomes)
median_incomes = df2_demo['hh_assets'].median()
print(median_incomes)
median_incomes = age_restricted_df1['hh_assets'].median()
print(median_incomes)
median_incomes = non_age_restricted_df1['hh_assets'].median()
print(median_incomes)
median_incomes = age_restricted_df2['hh_assets'].median()
print(median_incomes)
median_incomes = non_age_restricted_df2['hh_assets'].median()
print(median_incomes)

In [ ]:
median_incomes = age_restricted_merged['hh_assets'].median()
print(median_incomes)
median_incomes = non_age_restricted_merged['hh_assets'].median()
print( median_incomes)
median_incomes = age_restricted_merged['hh_income'].median()
print(median_incomes)
median_incomes = non_age_restricted_merged['hh_income'].median()
print(median_incomes)

In [ ]:
median_assets = merged_demo.groupby('race_simplified')['hh_assets'].median().sort_values(ascending=False)
print(median_assets)

In [ ]:
merged_demo['race_simplified'].value_counts()

In [ ]:
df2_demo['race_simplified'].value_counts()

In [ ]:
# 1. Create a new column that categorizes each applicant as White or Non-White
merged['race_group'] = merged['race_simplified'].apply(
    lambda x: 'White' if x == 'White' else 'Non-White'
)

# 2. Count applications to each town by race group
town_race_counts = merged.groupby(['property_town', 'race_group']).size().unstack(fill_value=0)

# 3. Calculate total applications per race group
race_totals = merged['race_group'].value_counts()

# 4. Calculate the percentage of each race group's applications that went to each town
town_race_percentages = (town_race_counts / race_totals) * 100

# 5. Round for readability
town_race_percentages = town_race_percentages.round(1)

# 6. Sort towns by highest percentages (priority on White, then Non-White)
town_race_percentages = town_race_percentages.sort_values(
    by=town_race_percentages.columns.tolist(), ascending=False
)

# 7. Display the result in a Jupyter notebook
from IPython.display import display
display(town_race_percentages)


In [ ]:
# Filter rows where property_town is 'ayer'
ayer_apps = merged[merged['property_town'] == 'North Andover']

# Calculate the median hh_income
median_income = ayer_apps['hh_income'].median()

# Print the result
print(f"The median household income for applications to North Andover is ${median_income:,.2f}")


In [ ]:
# Filter rows where property_town is 'ayer'
ayer_apps = age_restricted_merged[age_restricted_merged['property_town'] == 'Lakeville']

# Calculate the median hh_income
median_income = ayer_apps['hh_assets'].median()

# Print the result
print(f"The median household income for applications to Lakeville is ${median_income:,.2f}")


In [ ]:
# Filter rows where property_town is 'ayer'
ayer_apps = non_age_restricted_merged[non_age_restricted_merged['property_town'] == 'West Boylston']

# Calculate the median hh_income
median_income = ayer_apps['hh_assets'].median()

# Print the result
print(f"The median household income for applications to West Boylston is ${median_income:,.2f}")
